# Playwright with JavaScript — Day 3
## November 7, 2026 | Auto-waiting and Synchronization

`

### Learning objectives
- Explain why browser automation needs synchronization.
- Understand Playwright auto-waiting and actionability checks.
- Distinguish fixed waits from condition-based waits.
- Wait for elements, URL changes, popups and network responses.
- Diagnose flaky tests caused by timing problems.

> Playwright already waits automatically for many actions and retries web-first assertions. Add explicit waits only when synchronizing with a specific event or condition.

## 1. Why synchronization is needed

Web applications update asynchronously. Search results may arrive after an API call, a button may become enabled after validation, and a modal may appear after a click.

**Synchronization** coordinates the test with the application's state. A **race condition** occurs when the test and application progress at different speeds. A **flaky test** passes and fails inconsistently.

```javascript
await page.goto('https://example.com');
await page.locator('#search').fill('Playwright');
await page.locator('#searchButton').click();
console.log(await page.locator('.result').textContent());
```

The last line assumes the result is already present. That assumption may be false.

## 2. Playwright auto-waiting

Before actions such as `locator.click()`, Playwright waits for the target to be actionable. For a typical click it checks that the locator resolves to one element and that it is **visible, stable, receives events, and enabled**.

```javascript
const submitButton = page.locator('#submit');
await submitButton.click();
```

Usually, do not add a fixed delay first:

```javascript
await page.waitForTimeout(3000); // avoid as normal synchronization
await submitButton.click();
```

Auto-waiting does not mean every business operation is complete after a click. Verify the resulting UI state.

## 3. Actionability examples

```javascript
await page.getByRole('button', { name: 'Submit' }).click();
await page.locator('#email').fill('student@example.com');
await page.locator('#terms').check();
```

Different actions have different actionability requirements. For example, `click()` checks visibility, stability, event reception and enabled state; `fill()` checks visibility, enabled state and editability.

If an action times out, investigate the locator, visibility, enabled state, overlays, animations and page state. Do not immediately use `force: true` or increase the timeout.

## 4. Fixed wait vs condition-based wait

```javascript
await page.waitForTimeout(3000);
```

This always pauses for three seconds. It may waste time if the page is ready earlier or still be too short if the page is slow. Use fixed waits only for temporary debugging.

Condition-based locator wait:

```javascript
await page.locator('.success-message').waitFor({ state: 'visible' });
```

States: `attached`, `detached`, `visible`, `hidden`.

Often the clearest choice is a web-first assertion:

```javascript
await expect(page.locator('.success-message')).toBeVisible();
```

It retries until the assertion passes or times out.

## 5. Waiting for elements

```javascript
const message = page.locator('#status');
await message.waitFor({ state: 'visible' });
console.log(await message.textContent());

await page.locator('.loading-spinner').waitFor({ state: 'hidden' });
await expect(page.locator('.product-card')).toHaveCount(10);
```

If the goal is to verify visibility, hidden state or count, prefer a web-first assertion. Use `locator.waitFor()` when you specifically need to wait for a locator state before continuing.

## 6. Navigation and load states

```javascript
await page.goto('https://example.com');
await page.waitForLoadState('domcontentloaded');
await page.waitForLoadState('load');
await page.waitForLoadState('networkidle');
```

- `domcontentloaded`: the HTML has been parsed and DOMContentLoaded fired.
- `load`: the page load event fired.
- `networkidle`: network connections have been idle for a period.

Modern applications may continuously make background requests, so `networkidle` is not a universal signal that the app is ready. Prefer waiting for the specific UI state.

```javascript
await page.goto('https://example.com');
await expect(page.getByRole('heading')).toBeVisible();
```

## 7. Waiting for URL changes

Use `waitForURL()` when an action should navigate to a known destination.

```javascript
await Promise.all([
    page.waitForURL('**/about'),
    page.getByRole('link', { name: 'About' }).click()
]);
```

Register the wait before triggering the navigation so a fast event is not missed. `waitForNavigation()` is deprecated in current Playwright guidance for many scenarios; prefer `waitForURL()` when the expected destination is known.

## 8. Waiting for popups and new pages

```javascript
const popupPromise = page.waitForEvent('popup');
await page.getByRole('link', { name: 'Open popup' }).click();
const popup = await popupPromise;
await expect(popup).toHaveURL(/popup/);
```

For any new page in the browser context:

```javascript
const pagePromise = page.context().waitForEvent('page');
await page.getByRole('link', { name: 'Open new tab' }).click();
const newPage = await pagePromise;
await newPage.waitForLoadState();
console.log(await newPage.title());
```

The event wait is registered before the click.

## 9. Waiting for network responses

```javascript
const responsePromise = page.waitForResponse(response =>
    response.url().includes('/api/search') &&
    response.request().method() === 'GET' &&
    response.status() === 200
);

await page.getByRole('button', { name: 'Search' }).click();
const response = await responsePromise;
console.log('API status:', response.status());
```

Register the response wait before the action. A successful HTTP response does not prove the UI is correct; verify the user-visible result too:

```javascript
await expect(page.locator('.search-results')).toBeVisible();
```

## 10. Timeouts

```javascript
// Test timeout
 test.setTimeout(30000);

// Action timeout
await page.locator('#submit').click({ timeout: 10000 });

// Assertion timeout
await expect(page.locator('.success')).toBeVisible({ timeout: 10000 });
```

Configuration example:

```javascript
const { defineConfig } = require('@playwright/test');
module.exports = defineConfig({
    timeout: 30000,
    expect: { timeout: 5000 },
    use: { actionTimeout: 10000, navigationTimeout: 15000 }
});
```

Timeouts are maximum waiting limits, not guaranteed delays. Increasing one does not fix an incorrect locator or broken application.

## 11. Live coding: predictable dynamic UI

Create a file named `sync-demo.html` using the HTML below. It reveals a message after 1.5 seconds and loads three list items after 2 seconds.

```html
<!DOCTYPE html>
<html>
<head><title>Synchronization Practice</title></head>
<body>
  <h1>Synchronization Practice</h1>
  <button id="showMessage">Show message</button>
  <p id="message" hidden>Operation completed successfully</p>
  <button id="loadData">Load data</button>
  <p id="loading" hidden>Loading...</p>
  <ul id="results"></ul>
  <script>
    document.querySelector('#showMessage').addEventListener('click', () => {
      setTimeout(() => document.querySelector('#message').hidden = false, 1500);
    });
    document.querySelector('#loadData').addEventListener('click', () => {
      const loading = document.querySelector('#loading');
      const results = document.querySelector('#results');
      loading.hidden = false;
      results.innerHTML = '';
      setTimeout(() => {
        loading.hidden = true;
        results.innerHTML = '<li>Item A</li><li>Item B</li><li>Item C</li>';
      }, 2000);
    });
  </script>
</body>
</html>
```

Save this HTML in the same project directory as the test file. Then run the test below.

### Playwright test for the dynamic UI

```javascript
const { test, expect } = require('@playwright/test');
const path = require('path');

test('wait for delayed message', async ({ page }) => {
    await page.goto('file://' + path.resolve('sync-demo.html'));
    await page.locator('#showMessage').click();
    await expect(page.locator('#message')).toBeVisible();
    await expect(page.locator('#message'))
        .toHaveText('Operation completed successfully');
});

test('wait for loading to finish', async ({ page }) => {
    await page.goto('file://' + path.resolve('sync-demo.html'));
    await page.locator('#loadData').click();
    await expect(page.locator('#loading')).toBeVisible();
    await expect(page.locator('#loading')).toBeHidden();
    await expect(page.locator('#results li')).toHaveCount(3);
    await expect(page.locator('#results')).toContainText('Item B');
});
```

**Discuss:** Why is a fixed two-second wait unnecessary? What does `toBeHidden()` verify? Why is `toHaveCount(3)` safer than immediately reading the list?

## 12. Practice using the course website

Practice site: https://testautomationpractice.blogspot.com/

Public practice sites can change. Inspect the live DOM and confirm selectors before using them; do not assume a selector or result element exists.

```javascript
const { test, expect } = require('@playwright/test');

test('demonstrate auto-waiting and assertions', async ({ page }) => {
    await page.goto('https://testautomationpractice.blogspot.com/');
    const nameInput = page.locator('#name');
    await expect(nameInput).toBeVisible();
    await nameInput.fill('Arun');
    await expect(nameInput).toHaveValue('Arun');
});
```

Instructor demonstration:
1. Run the test normally.
2. Add a short fixed wait temporarily and compare.
3. Remove the fixed wait.
4. Demonstrate `toBeVisible()` and `locator.waitFor()`.
5. Use an incorrect selector, inspect the timeout error, then correct it.

## 13. Common mistakes and better alternatives

| Avoid | Prefer | Reason |
|---|---|---|
| Fixed 5-second wait before every click | Let the action auto-wait | Less delay, fewer timing assumptions |
| Read text immediately after async action | `expect(locator).toHaveText(...)` | Retries until expected text appears |
| Listen for popup after clicking | Register event wait before click | Avoid missing a fast event |
| Assume `networkidle` means app is ready | Wait for relevant UI state | Background requests may continue |
| Raise every timeout excessively | Diagnose locator and page state | Avoid hiding real problems |

Immediate check vs retrying assertion:

```javascript
const visible = await page.locator('#message').isVisible(); // immediate Boolean
await expect(page.locator('#message')).toBeVisible();       // retries
```

## 14. Student assignment — 30 minutes

### Task A: Auto-waiting
1. Open the synchronization demo page.
2. Click **Show message**.
3. Verify the message becomes visible and has the exact text.

### Task B: Loading indicator
1. Click **Load data**.
2. Verify loading becomes visible.
3. Verify loading becomes hidden.
4. Verify exactly three list items appear.
5. Verify the list contains `Item B`.

### Task C: Compare approaches
Write two tests: one using `waitForTimeout()` and one using a web-first assertion. Explain which is more reliable and why.

### Task D: Debugging
Change `#message` to `#messages` in one assertion. Run the test, inspect the error, identify the cause, correct the selector and rerun.

**Submission:** one `.spec.js` file, execution screenshot or terminal result, and short answers to the discussion questions.

## 15. Quick knowledge check

1. What is auto-waiting?
2. Name four actionability checks used by `click()`.
3. Why is `waitForTimeout(5000)` usually poor synchronization?
4. Difference between `locator.waitFor()` and `toBeVisible()`?
5. What does `domcontentloaded` mean?
6. Why register a popup wait before clicking?
7. When is `waitForResponse()` useful?
8. Does a successful API response prove the UI is correct?
9. Difference between `isVisible()` and `toBeVisible()`?
10. Does increasing timeout fix an incorrect selector?

### Answer key
1. Playwright waits for required conditions before supported actions.
2. Visible, stable, receives events, enabled.
3. It waits a fixed duration, which may be too long or too short.
4. Locator wait waits for a locator state; the assertion retries while verifying an expectation.
5. The HTML has been parsed and DOMContentLoaded fired.
6. To avoid missing a fast event.
7. When an action triggers a request whose response matters.
8. No; verify the visible UI too.
9. `isVisible()` is immediate; `toBeVisible()` retries.
10. No.

## 16. Day 3 recap and references

- Browser applications update asynchronously.
- Synchronization prevents tests from relying on timing assumptions.
- Playwright auto-waits for actionability before supported actions.
- Web-first assertions retry and clearly verify UI state.
- Use explicit waits for meaningful events or conditions: URL changes, popups, responses or locator states.
- Avoid fixed sleeps in normal tests.
- `networkidle` is not a universal signal that an app is ready.
- A timeout is a maximum wait limit, not a guaranteed delay.

### Official documentation
- Auto-waiting: https://playwright.dev/docs/actionability
- Assertions: https://playwright.dev/docs/test-assertions
- Page API: https://playwright.dev/docs/api/class-page
- Locator API: https://playwright.dev/docs/api/class-locator
- Navigation: https://playwright.dev/docs/navigations

### Next class — Day 4
**Dropdowns, tables and web controls**